# CranioVision — nnU-Net Training

**Branch:** `feature/nnU-Net`  
**Model:** MONAI DynUNet with nnU-Net-style hyperparameters (Option B)  
**Dataset:** BraTS 2024 small (200 cases, 140 train)  
**Runtime target:** Kaggle T4 GPU (~2-3 hours for 100 epochs)

---

## What makes nnU-Net different from Attention U-Net / SwinUNETR?

- **Residual blocks** in encoder and decoder (vs plain conv in Attention U-Net)
- **Instance normalization** + Leaky ReLU (vs BatchNorm/ReLU)
- **Wider architecture** — 32/64/128/256/320/320 filters across 6 levels
- **No attention gates** (simpler, faster than Attention U-Net)
- **No transformer** (lighter than SwinUNETR)

In our ensemble this model provides the **"conservative baseline"** — nnU-Net's strength is being reliably good everywhere, rarely failing catastrophically. It complements Attention U-Net (high recall) and SwinUNETR (high precision on small structures).

## 1. Environment Setup

In [ ]:
import sys
import os
from pathlib import Path

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    PROJECT_ROOT = Path('/kaggle/working/CranioVision')
    IS_KAGGLE = True
else:
    PROJECT_ROOT = Path.cwd().parent
    IS_KAGGLE = False

sys.path.insert(0, str(PROJECT_ROOT))

print(f'Environment : {"KAGGLE" if IS_KAGGLE else "LOCAL"}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Exists      : {PROJECT_ROOT.exists()}')

In [ ]:
if IS_KAGGLE:
    import src.cranovision.config as cfg

    kaggle_input = Path('/kaggle/input')
    candidates = list(kaggle_input.rglob('BraTS2024_small_dataset'))
    candidates = [c for c in candidates if list(c.glob('BraTS-*'))]

    if not candidates:
        for d in kaggle_input.rglob('BraTS-GLI-*'):
            if d.is_dir():
                candidates = [d.parent]
                break

    if candidates:
        cfg.RAW_DATA_DIR = candidates[0]
        n_cases = len(list(cfg.RAW_DATA_DIR.glob('BraTS-*')))
        print(f'✅ Kaggle dataset detected: {cfg.RAW_DATA_DIR}')
        print(f'   Cases: {n_cases}')
    else:
        raise FileNotFoundError('Could not locate BraTS dataset under /kaggle/input.')

    cfg.MODELS_DIR  = Path('/kaggle/working/models')
    cfg.OUTPUTS_DIR = Path('/kaggle/working/outputs')
    cfg.SPLITS_DIR  = Path('/kaggle/working/splits')
    for d in (cfg.MODELS_DIR, cfg.OUTPUTS_DIR, cfg.SPLITS_DIR):
        d.mkdir(parents=True, exist_ok=True)
    print(f'Models dir  : {cfg.MODELS_DIR}')
    print(f'Outputs dir : {cfg.OUTPUTS_DIR}')

In [ ]:
from src.cranovision.config import (
    RAW_DATA_DIR, MODELS_DIR, OUTPUTS_DIR, DEVICE, USE_AMP,
    PATCH_SIZE, BATCH_SIZE, NUM_WORKERS, PIN_MEMORY,
    LEARNING_RATE, WEIGHT_DECAY, MAX_EPOCHS, VAL_INTERVAL,
    NUM_CLASSES, print_config,
)
from src.cranovision.data import (
    get_splits, get_train_transforms, get_val_transforms,
)
from src.cranovision.models import build_nnunet_for_training, count_parameters
from src.cranovision.training import TrainConfig, train

print_config()

## 2. Data Loading (SAME split as other 2 models)

In [ ]:
train_files, val_files, test_files = get_splits()

In [ ]:
from monai.data import CacheDataset, DataLoader

train_transforms = get_train_transforms()
val_transforms   = get_val_transforms()

print('Building CacheDatasets...')

train_ds = CacheDataset(data=train_files, transform=train_transforms,
                         cache_rate=1.0, num_workers=NUM_WORKERS)
val_ds   = CacheDataset(data=val_files,   transform=val_transforms,
                         cache_rate=1.0, num_workers=NUM_WORKERS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                           num_workers=0, pin_memory=PIN_MEMORY)

print(f'\n✅ Loaders ready.')
print(f'  Train batches: {len(train_loader)}')
print(f'  Val batches  : {len(val_loader)}')

## 3. Model — nnU-Net-style DynUNet

Using nnU-Net's standard brain MRI filter config: `(32, 64, 128, 256, 320, 320)`.  
If OOM on Kaggle, reduce to `(16, 32, 64, 128, 256, 256)`.

In [ ]:
model = build_nnunet_for_training(
    filters=(32, 64, 128, 256, 320, 320),
    deep_supervision=False,              # ← simpler, avoids wrapper complexity
    dropout=0.1,
).to(DEVICE)

info = count_parameters(model)
print(f'nnU-Net (DynUNet) built on {DEVICE}')
print(f'  Total params    : {info["total"]:>12,}  ({info["total_M"]:.2f}M)')
print(f'  Trainable params: {info["trainable"]:>12,}')

import torch
with torch.no_grad():
    dummy = torch.randn(1, 4, *PATCH_SIZE).to(DEVICE)
    out = model(dummy)
print(f'\nShape check:')
print(f'  Input  : {tuple(dummy.shape)}')
print(f'  Output : {tuple(out.shape)}')
del dummy, out
torch.cuda.empty_cache()

## 4. Loss, Optimizer, Scheduler

In [ ]:
from monai.losses import DiceCELoss

loss_fn = DiceCELoss(
    to_onehot_y=True,
    softmax=True,
    lambda_dice=1.0,
    lambda_ce=1.0,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
    eta_min=1e-6,
)

print(f'Loss     : DiceCELoss')
print(f'Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'Scheduler: CosineAnnealingLR (T_max={MAX_EPOCHS})')

## 5. Train

In [ ]:
train_config = TrainConfig(
    model_name='nnunet',
    max_epochs=MAX_EPOCHS,
    val_interval=VAL_INTERVAL,
    patch_size=PATCH_SIZE,
    use_amp=False,                      # ← disabled: DynUNet has AMP incompat with MONAI 1.5.2
    ckpt_dir=MODELS_DIR,
    history_dir=OUTPUTS_DIR,
)

history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    config=train_config,
)

## 6. Plot Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f0f')
for ax in axes:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for s in ax.spines.values():
        s.set_edgecolor('#444')

axes[0].plot(range(1, len(history.train_loss) + 1), history.train_loss,
              color='#FF6B35', linewidth=2, label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('DiceCE Loss')
axes[0].set_title('nnU-Net — Training Loss')
axes[0].legend(facecolor='#2a2a2a', labelcolor='white')
axes[0].grid(alpha=0.2)

axes[1].plot(history.val_epochs, history.val_dice,
              color='#4ECDC4', linewidth=2, marker='o', markersize=5,
              label='Mean Val Dice')
axes[1].axhline(y=history.best_dice, color='#FFD700', linestyle='--',
                 alpha=0.7, label=f'Best: {history.best_dice:.4f} (ep {history.best_epoch})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Dice (excl. background)')
axes[1].set_title('nnU-Net — Validation Dice')
axes[1].legend(facecolor='#2a2a2a', labelcolor='white')
axes[1].grid(alpha=0.2)
axes[1].set_ylim(0, 1)

plt.suptitle('CranioVision — nnU-Net Training',
              color='white', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'nnunet_curves.png', dpi=150,
             bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print(f'Saved: {OUTPUTS_DIR / "nnunet_curves.png"}')

## 7. Summary

In [ ]:
print('=' * 60)
print('nnU-Net — TRAINING COMPLETE')
print('=' * 60)
print(f'Best epoch     : {history.best_epoch}')
print(f'Best Val Dice  : {history.best_dice:.4f}')
print(f'Total epochs   : {MAX_EPOCHS}')
print(f'Checkpoint     : {MODELS_DIR / "nnunet_best.pth"}')
print(f'History JSON   : {OUTPUTS_DIR / "nnunet_history.json"}')
print(f'Curves PNG     : {OUTPUTS_DIR / "nnunet_curves.png"}')
print('=' * 60)